# 01 · Exploratory Data Analysis — drowsiness feature windows

Explores the window-level features defined in `ml/feature_spec.json`: class balance, per-feature distributions by class, correlations and a 2-D PCA projection.

> Run from the repository root (`jupyter notebook ml/notebooks`). If you have not collected real data yet, run `python -m ml.preprocessing.synthetic` first — and remember that synthetic data only demonstrates the pipeline.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from ml.config import FEATURES, LABEL, GROUP
from ml.preprocessing.build_dataset import load_raw, clean

sns.set_theme(style="whitegrid")
df, report = clean(load_raw())
report

## Class balance and sessions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df[LABEL].map({0: "alert", 1: "drowsy"}).value_counts().plot.bar(ax=ax[0], color=["#06b6d4", "#f87171"], title="Windows per class")
df.groupby(GROUP)[LABEL].mean().plot.bar(ax=ax[1], title="Drowsy fraction per session")
plt.tight_layout()

## Feature distributions by class

Good features separate the two histograms. PERCLOS and longest closure are expected to be the strongest single indicators.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, f in zip(axes.flat, FEATURES):
    sns.kdeplot(data=df, x=f, hue=LABEL, common_norm=False, fill=True, ax=ax, palette=["#06b6d4", "#f87171"])
axes.flat[-1].axis("off")
plt.tight_layout()

## Univariate effect sizes (Cohen’s d)

In [ ]:
def cohens_d(a, b):
    s = np.sqrt(((len(a)-1)*a.var() + (len(b)-1)*b.var()) / (len(a)+len(b)-2))
    return (b.mean() - a.mean()) / s

effect = pd.Series({f: cohens_d(df.loc[df[LABEL]==0, f], df.loc[df[LABEL]==1, f]) for f in FEATURES}).sort_values(key=abs, ascending=False)
effect.plot.barh(figsize=(7, 4), color=np.where(effect > 0, "#f87171", "#06b6d4"), title="Cohen’s d (drowsy − alert)")
effect.round(3)

## Correlation matrix

Highly correlated features (e.g. PERCLOS ↔ EAR mean) carry redundant information; regularised models handle this, but it matters when interpreting coefficients.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Pearson correlation")

## PCA projection

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Z = PCA(n_components=2).fit(StandardScaler().fit_transform(df[FEATURES]))
proj = Z.transform(StandardScaler().fit_transform(df[FEATURES]))
plt.figure(figsize=(7, 5))
plt.scatter(proj[:, 0], proj[:, 1], c=df[LABEL], cmap="coolwarm", s=6, alpha=0.5)
plt.xlabel(f"PC1 ({Z.explained_variance_ratio_[0]:.0%})"); plt.ylabel(f"PC2 ({Z.explained_variance_ratio_[1]:.0%})")
plt.title("Windows in PCA space (blue = alert, red = drowsy)")

## Next steps

`python -m ml.preprocessing.build_dataset` → `python -m ml.training.train_classifier` → see `ml/evaluation/reports/`.